# Cancer Data Logistic Regression (Student)

In [ ]:
import pandas as pd
import os
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import r2_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from scipy.stats import chi2_contingency


In this notebook we'll analyze the [Wisconsin Breast Cancer](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) dataset. Features are computed from a digitized image of a possibly cancerous mass. Each observation is labeled 'B' or 'M' for 'benign' or 'malignant'.

In [ ]:
## if you have it locally
data_root = "../data" ## change this line if needed
filename = "Cancer_Data_Cleaned.csv"
filepath = os.path.join(data_root, filename)

## url for download
url  = "https://aet-cs.github.io/white/ML/data/Cancer_Data_Cleaned.csv"

# use url or filepath
df = pd.read_csv(url)

In [ ]:
df.dtypes

We define a helper function that drops 'se' and 'worst' features -- standard error and extreme values -- and keep only the means from each observation.

In [ ]:
def get_data(filename):
    df = pd.read_csv(filename)
    df.drop(['id', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst'], axis=1, inplace=True)
    return df

In [ ]:
df = get_data(url)
df

In [ ]:
df.describe()

A basic preprocessor to handle missing values, infinities, and NaNs, and one_hot_encoding (only applicable here to the binary feature)

In [ ]:
def pre_process_data(df, one_hot_encode = False):
    # Use sklearn Imputers to fill in the categorical and numerical columns
    simple_median = SimpleImputer(strategy='median')
    simple_most_freq = SimpleImputer(strategy='most_frequent')

    num_cols = df.select_dtypes(include=np.number).columns # numerical data
    cat_cols = df.select_dtypes(include=["object", "string"]).columns # categorical data

    # Handle missing values
    df[num_cols] = simple_median.fit_transform(df[num_cols])
    df[cat_cols] = simple_most_freq.fit_transform(df[cat_cols])

    # Replace infinite values with NaN and then impute
    df = df.replace([np.inf, -np.inf], np.nan)
    df[num_cols] = simple_median.fit_transform(df[num_cols])

    if one_hot_encode:
        O_encoder = OrdinalEncoder()
        df[cat_cols]= O_encoder.fit_transform(df[cat_cols])

    return df

And a method get to a train and test split, after applying normalization (the scaler). In this example we scale based on *median* and *IQR* instead of *mean* and *standard deviation*

In [ ]:
def get_test_train(df, test_size = 0.2, random_state = True):
    target = "diagnosis"
    X = df.drop(target, axis=1)

    # Robust scaling to handle outliers, based on median and inter-quartile range
    scaler = preprocessing.RobustScaler().fit(X)
    X = scaler.transform(X)

    # Additional check for any remaining infinite values
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    y = df[target]
    if random_state is True:
        X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=test_size)
    else:
        X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=test_size, random_state=42)
    return X_train, X_test, y_train, y_test

## Start Work Here

In [ ]:
# These lines would load the data locally
data_root = "../data"
filename = "cancer_data_cleaned.csv"
filepath = os.path.join(data_root, filename)

# Perform a logistic regression
df = get_data(url)
df = pre_process_data(df, one_hot_encode = True)
X_train, X_test, y_train, y_test = get_test_train(df, random_state = False)

# Create logistic regression with liblinear solver
lreg = LogisticRegression(
    C=1.0,
    solver='liblinear',  # Different solver that should be more stable than the default
    max_iter=10000,  # increased from default because of squirrely data
    tol=1e-4, # tolerance set to 0.0001
)
model = lreg.fit(X_train, y_train)

In [ ]:
X_test

Get the model score

In [ ]:
pred = lreg.predict(X_test[0:1])
print("Prediction for first sample:", pred[0])
print(f"Train accuracy  = {model.score(X_train, y_train):.3}")
print(f"Test  accuracy  = {model.score(X_test, y_test):.3}")

In [ ]:
# Get feature names from the original dataframe (excluding the target column)
feature_names = df.drop('diagnosis', axis=1).columns

# Create a DataFrame with feature names and their corresponding coefficients
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lreg.coef_[0],
    'Abs_Coefficient': abs(lreg.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("Model Coefficients (sorted by absolute value):")
print(coef_df)

print("\nIntercept:", lreg.intercept_[0])

Quick snapshot of the confusion matrix (rows are truth  0/1 and cols are predictions 0/1). You can shift-tab on the parens the see the method signature)

In [ ]:
confusion_matrix(y_pred = model.predict(X_test), y_true = y_test)

Here we get the raw probabilites from X-test. So we want real values in (0,1). Higher values *should* correspond to positive observations. The model rounds these probabilities to 0 or 1 when it makes its prediction. We're interested in what rule we use to round.

In [ ]:
# Get predicted probabilities for the test data
y_prob = model.predict_proba(X_test)[:,1]

If we sort y_prob and y_test in the same order, then we can make an interesting plot

In [ ]:
# Sort the data for plotting
y_pred = model.predict(X_test)
sorted_indices = np.argsort(y_prob)  ## argsort returns the indices ordered by the key values, so we can copy the sort order around
sorted_y_prob = y_prob[sorted_indices]
sorted_y_test = np.array(y_test)[sorted_indices]
sorted_y_pred = np.array(y_pred)[np.argsort(y_pred)]

Print the vector of sorted_y_prob and of sorted_y_test to verify the are generally increasing from 0 to 1. We create a dataframe also just because.

In [ ]:
pd.set_option('display.max_rows', None)

results_df = pd.DataFrame({"Probs": sorted_y_prob,
                           "True y": sorted_y_test,
                           "Pred y":  sorted_y_pred})
results_df


Now we can plot the logistic model, along with the true and predicted values. We set the split point at 0.5, but it can be changed

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# find the index of the first value in y_prob that is greater than split_point
split_point = 0.95
index = np.searchsorted(sorted_y_prob, split_point)
new_y_pred = np.where(sorted_y_prob > split_point, 1, 0)

x_test = np.array(range(len(y_test))) ## just a counter
# For demonstration if needed:
# x_test = np.linspace(-3, 3, len(y_test))

# Compute confusion categories
# these numpy calculation return a LIST of INDICES, which is very powerful!
# see how they're used below
tp = (sorted_y_test == 1) & (new_y_pred == 1)
tn = (sorted_y_test == 0) & (new_y_pred == 0)
fp = (sorted_y_test == 0) & (new_y_pred == 1)
fn = (sorted_y_test == 1) & (new_y_pred == 0)

# --- Plot setup ---
plt.figure(figsize=(12 ,5))

# Logistic curve (predicted probabilities)
plt.plot(x_test, sorted_y_prob, color='black', linewidth=2, label='Predicted probability')

# Scatter for each category
plt.scatter(x_test[tp], new_y_pred[tp], color='blue', s=40, label='True Positive')
plt.scatter(x_test[tn], new_y_pred[tn], color='green', s=40, label='True Negative')
plt.scatter(x_test[fp], new_y_pred[fp], color='red', s=40, label='False Positive')
plt.scatter(x_test[fn], new_y_pred[fn], color='orange', s=40, label='False Negative')

# Optional: plot your custom split point threshold line
plt.axhline(split_point, color='gray', linestyle='--', label=f'Split point = {split_point:.2f}')

# Labels and legend
plt.xlabel('x index')
plt.ylabel('Probability / True label')
plt.title('Logistic Regression Classification Outcomes')
plt.legend(loc='center left', frameon=True)

plt.show()


# Binary Classification Helper Functions

Binary classification for logistic regression relies on knowing where the **split point** (threshold) is.
By default, this split point is `alpha = 0.5`, meaning that predicted probabilities greater than 0.5
are classified as `1` (positive), and those less than or equal to 0.5 are classified as `0` (negative).

However, `alpha = 0.5` is not always optimal — depending on the dataset and the cost of errors,
a different threshold may produce better results.

In this exercise, you will define a set of helper functions to analyze binary classification results
and help determine the optimal value of `alpha`.  

Each function will help you compute parts of the **confusion matrix** and related performance metrics.

---

## Your tasks

1. **binary_classify(y_prob, alpha)**  
   - Given a vector of predicted probabilities (`y_prob`), return a vector of 0s and 1s  
     depending on whether each probability exceeds `alpha`.

2. **tp(true, observed)**  
   - Return the number of *true positives*: cases where both the true label and the prediction are 1.

3. **tn(true, observed)**  
   - Return the number of *true negatives*: cases where both are 0.

4. **fp(true, observed)**  
   - Return the number of *false positives*: predicted 1 but actually 0.

5. **fn(true, observed)**  
   - Return the number of *false negatives*: predicted 0 but actually 1.

6. **precision(true, observed)**  
   - Compute precision = TP / (TP + FP).  
     (Handle divide-by-zero cases gracefully.)

7. **recall(true, observed)**  
   - Compute recall = TP / (TP + FN).  
     (Handle divide-by-zero cases gracefully.)

8. **f1(true, observed, weights)**  
   - Return a *f1* using  (TP, FP, TN, FN).  

---

Once you’ve implemented these, you’ll be able to:
- Explore how changing `alpha` affects precision, recall, and F1 score,  
- Visualize the trade-off between sensitivity and specificity, and  
- Find the threshold that maximizes your chosen performance metric.


In [ ]:
import numpy as np

def binary_classify(y_prob, alpha):
    """Return 1 if probability > alpha, else 0."""

def tp(true, observed):
    """True positives: predicted 1, actually 1."""


def tn(true, observed):
    """True negatives: predicted 0, actually 0."""


def fp(true, observed):
    """False positives: predicted 1, actually 0."""


def fn(true, observed):
    """False negatives: predicted 0, actually 1."""


def precision(true, observed):
    """Precision = TP / (TP + FP)"""

def recall(true, observed):
    """Recall = TP / (TP + FN)"""

def f1(true, observed):
    """F1 = 2 * (precision * recall) / (precision + recall)"""


## Step 1: Test your classification and precision functions

Before experimenting with different thresholds (`alpha`), let's verify that your helper
functions are working correctly.

We'll start by classifying predictions using the default split point `alpha = 0.5`,
which is standard for logistic regression. Then we'll measure **precision**, which tells us:

> Of all the points predicted as positive (1), how many were actually positive?

A high precision means your model makes few false positives.

In [ ]:
observed = binary_classify(sorted_y_prob, 0.5)
q = zip(observed, sorted_y_test)
print(precision(sorted_y_test, observed))

## Step 2: Precision–Recall Trade-off

In binary classification, changing the threshold `α` (alpha) controls how “strict” your model is
when deciding what counts as a positive prediction.

- A **low α** means you label more cases as positive → higher recall, lower precision.
- A **high α** means you label fewer cases as positive → lower recall, higher precision.

In this exercise, you’ll explore this trade-off by varying `α` from 0 to 1 (in steps of 0.01) and plotting
the resulting **precision** and **recall** values.

Your goal: complete the loop below to compute `precision(α)` and `recall(α)` for each threshold,
and then visualize them on a scatter plot.

In [ ]:
ps = [] #empty list of precisions
rs = [] #empty list of recalls
for i in range(100):
    alpha = i/100.0
    # compute results
    # and add values to the precisions and recalls vectors


# --- Plot precision vs recall ---
plt.figure(figsize=(7, 5))
plt.plot(ps, rs, marker='o', markersize=3, linestyle='-', color='royalblue')
plt.xlabel("Precision")
plt.ylabel("Recall")
plt.title("Precision–Recall Curve for Varying α (Threshold)")
plt.grid(True, linestyle='--', alpha=0.6)

# --- Annotate alpha values at selected points ---
for i in range(0, 100, 10):  # Label every 10th alpha value
    alpha = i / 100.0
    # Offset labels slightly for readability
    plt.annotate(f'α={alpha:.2f}',
                 (ps[i], rs[i]),
                 textcoords="offset points",
                 xytext=(8, -8),
                 ha='left', fontsize=8,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.6, lw=0))

plt.show()

## Step 3: Reflect on what you see

Look carefully at your Precision–Recall plot. Think about how changing α (the classification threshold)
affects your model’s behavior.

### Reflection Questions

1. What general pattern do you notice between **precision** and **recall** as α increases?
   - Why does this happen?

2. Where on the plot do you see the “best balance” between precision and recall?
   - How would you decide what balance is best for a particular application?

3. Suppose your model is being used for **disease detection** or **fraud detection**:
   - Would you prefer higher precision or higher recall? Why?

4. Did α = 0.5 (the default) seem like a good threshold for this dataset?
   - If not, what would you change it to, and why?

5. What happens near the extreme α values (very small or very large)?
   - How does that relate to how often the model predicts “positive”?

Take a moment to discuss or write down your thoughts before moving on.


Now we want to see how weighted F1-scores vary as a function of alpha.

###  Visualization

Now, let’s **plot all three metrics** precision, recall, f1 score, versus the classification threshold to see what’s going on.

In [ ]:
# Thresholds to test
thresholds = np.linspace(0, 1, 200)

# Calculate precision, recall, F1 for each threshold
precisions = []
recalls = []
f1_scores = []

for t in thresholds:
    ### make 3 vectors to plot below

# Plot
plt.figure(figsize=(7, 5))
plt.plot(thresholds, precisions, label="Precision")
plt.plot(thresholds, recalls, label="Recall")
plt.plot(thresholds, f1_scores, label="F1 Score", linewidth=2)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision, Recall, and F1 vs. Classification Threshold")
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.grid(True, alpha=0.3)
plt.show()

## Step 4: Finding the optimal split-point

**Follow-up (next cell)**
In the next step, we’ll use `numpy` to *find the threshold* that gives the **maximum F1 score**.

In [ ]:
np.argmax(f1_scores)

**The best alpha**
Based on the previous result, determine the best alpha and fill it in below for opt_alpha

In [ ]:
opt_alpha = ###

observed = binary_classify(y_prob, opt_alpha)
ps = precision(y_test, observed)
rs = recall(y_test, observed)
opt_tp = tp(y_test, observed)
opt_tn = tn(y_test, observed)
opt_fp = fp(y_test, observed)
opt_fn = fn(y_test, observed)

Now draw a nice confusion matrix.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, observed, labels=lreg.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=lreg.classes_)
disp.plot()
plt.show()